# Building a Simple Phase Diagram

This example shows how the `gauge_fix` and `legendre_transform` functions combine to form a simple T-x phase diagram. 
We construct a minimal binary system with two ideal phases (Liquid and Solid) where the pure component reference states are temperature dependent.


In [ ]:
import sys
from pathlib import Path
# Add src directories to path for local execution
sys.path.insert(0, str(Path().resolve().parent / 'src'))
sys.path.insert(0, str(Path().resolve().parent.parent / 'thermograph' / 'src'))

import torch
from zgraph import *
import plotly.graph_objects as go

# Helper to detach PyTorch tensors
from torch.utils._pytree import tree_map
def to_numpy(pytree):
    return tree_map(lambda x: x.detach().cpu().numpy().squeeze(), pytree)


In [ ]:
# Define coordinates: T, mu1, mu2
T, mu1, mu2 = SignalNodes(0, 1, 2)
R = 8.314
RT = FactorNode([[R]], [T])

# Ideal mixing phases with temperature-dependent references
# We set Liquid as the high-temperature reference: G_L = 0 for both components.
# Solid is stable at low temperatures.
# Component 1: Tm = 1000 K, dS = -10 J/mol.K -> dH = -10000 J/mol
# Component 2: Tm = 1500 K, dS = -10 J/mol.K -> dH = -15000 J/mol
# Free energy of melting: G_S - G_L = dH - T*dS

# Reference energy nodes for the Solid phase (G_L is implicitly 0)
def g_s_1(x):
    return -10000.0 + 10.0 * x[0]

def g_s_2(x):
    return -15000.0 + 10.0 * x[0]

G1_S = DynamicLeafNode(g_s_1, [0])
G2_S = DynamicLeafNode(g_s_2, [0])

# Effective chemical potentials (w = mu - G_ref)
# Liquid (G_ref = 0):
w1_L = FactorNode([[1.0]], [mu1])
w2_L = FactorNode([[1.0]], [mu2])

# Solid: w = mu - G_S
w1_S = FactorNode([[1.0, -1.0]], [mu1, G1_S])
w2_S = FactorNode([[1.0, -1.0]], [mu2, G2_S])

# Phases: logsumexp over components (ideal mixing)
phase_L = FactorNode(torch.eye(2), [w1_L, w2_L], beta=RT)
phase_S = FactorNode(torch.eye(2), [w1_S, w2_S], beta=RT)

# Total System: equilibrium is the minimum of the two phases (SoftMin with beta->0)
system = FactorNode(torch.eye(2), [phase_L, phase_S], beta=0.0)

fcns = [phase_L, phase_S, system]


## Find the Equilibrium Manifold (Gauge Fix)
Since ZGraph natively operates in implicit state variables (chemical potentials), our first step is to constrain the system to the equilibrium manifold.
We create a mesh over Temperature and chemical potential difference, and use `gauge_fix` to shift the coordinates such that the total system grand potential is exactly 0.


In [ ]:
# Compile the base graphs for execution
fcns_compiled = graph_to_function(fcns, compile=True)

# Create inputs
T_vals = torch.linspace(500, 2000, 150)
mu_diff = torch.linspace(-100000, 100000, 150)

T_grid, mu_grid = torch.meshgrid(T_vals, mu_diff, indexing='ij')

# Since it's a binary system, we parameterize by the difference in chemical potentials.
# We set mu1 = mu_grid/2, mu2 = -mu_grid/2
inputs = torch.stack([T_grid, mu_grid/2, -mu_grid/2], dim=-1)

# Flatten inputs for the batched compiled function
inputs_flat = inputs.view(-1, 3)

# Gauge Fix: project onto equilibrium manifold
shifted_inputs_flat = gauge_fix(fcns_compiled[2], inputs_flat, [1, 2])


## Legendre Transform
With our coordinates safely anchored to the equilibrium manifold, we can now apply the Legendre transform to map the chemical potentials `[mu1, mu2]` (indices 1, 2) to their conjugate mole fractions `[x1, x2]`.


In [ ]:
lt_fcns = legendre_transform(fcns, [1, 2])
lt_fcns_compiled = graph_to_function(lt_fcns, compile=True)

# Evaluate Legendre transforms on the gauge-fixed inputs
# Returns (free_energy, dual_coords) where dual_coords are [T, x1, x2]
system_lt_vals_flat = lt_fcns_compiled[2](shifted_inputs_flat)
free_energy_flat, dual_coords_flat = to_numpy(system_lt_vals_flat)

free_energy = free_energy_flat.reshape(150, 150)
dual_coords = dual_coords_flat.reshape(150, 150, 3)


## T-x Phase Diagram
By sweeping the chemical potential difference at various temperatures and mapping it to the conjugate mole fractions via the Legendre transform, the phase boundaries naturally emerge as gaps in the composition space.

We can also use the domain-specific `PhaseDiagramCompiler` from the `thermograph` package to directly extract the precise boundary lines (the tie-lines) using `zgraph`'s numerical solvers.


In [ ]:
# Extract explicit Phase Boundaries using Thermograph
from thermograph.visualize import PhaseDiagramCompiler

# The compiler needs the uncompiled root graph
compiler = PhaseDiagramCompiler(system)

# We pass our 2D grid: [Batch_T, Sweep_Mu, N_Signals]
# The solver will search for the boundary along the Sweep_Mu dimension!
batched_x = compiler.extract_boundaries(inputs, mu_index=1)

# batched_x has shape [Batch_T, 2] representing the two tie-line compositions at each temperature
x_liquidus = batched_x[:, 0].detach().cpu().numpy()
x_solidus = batched_x[:, 1].detach().cpu().numpy()


In [ ]:
import numpy as np
T_vals_np = T_vals.numpy()

# Ensure consistent left/right ordering of the extracted boundaries
x_left = np.minimum(x_liquidus, x_solidus)
x_right = np.maximum(x_liquidus, x_solidus)

fig = go.Figure()

# 1. Solid Region Polygon (from x=0 to x_left)
fig.add_trace(go.Scatter(
    x=np.concatenate([np.zeros_like(x_left), x_left[::-1]]),
    y=np.concatenate([T_vals_np, T_vals_np[::-1]]),
    fill='toself',
    fillcolor='rgba(240, 128, 128, 0.5)', # lightcoral with opacity
    line=dict(color='rgba(255,255,255,0)'),
    name='Solid Region'
))

# 2. Liquid Region Polygon (from x_right to x=1.0)
fig.add_trace(go.Scatter(
    x=np.concatenate([x_right, np.ones_like(x_right)]),
    y=np.concatenate([T_vals_np, T_vals_np[::-1]]),
    fill='toself',
    fillcolor='rgba(173, 216, 230, 0.5)', # lightblue with opacity
    line=dict(color='rgba(255,255,255,0)'),
    name='Liquid Region'
))

# 3. Two-Phase Region Polygon (between x_left and x_right)
fig.add_trace(go.Scatter(
    x=np.concatenate([x_left, x_right[::-1]]),
    y=np.concatenate([T_vals_np, T_vals_np[::-1]]),
    fill='toself',
    fillcolor='rgba(211, 211, 211, 0.5)', # lightgray with opacity
    line=dict(color='rgba(255,255,255,0)'),
    name='L + S Region'
))

# Overlay the bold boundary lines on top
fig.add_trace(go.Scatter(x=x_left, y=T_vals_np, mode='lines', name='Solidus', line=dict(color='red', width=3)))
fig.add_trace(go.Scatter(x=x_right, y=T_vals_np, mode='lines', name='Liquidus', line=dict(color='blue', width=3)))

fig.update_layout(
    title='Binary T-x Phase Diagram (Ideal Mixing)',
    xaxis_title='Mole Fraction Component 1 (x1)',
    yaxis_title='Temperature (K)',
    xaxis_range=[-0.05, 1.05],
    yaxis_range=[500, 2000],
    width=700,
    height=600,
    plot_bgcolor='white',
)
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray', zeroline=False)
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray', zeroline=False)

fig.show()


## 3D Gibbs Free Energy Surface
We can visualize the underlying convex energy landscape that drives this phase diagram. The Legendre dual of the `raw_phi` potential is the Gibbs Free Energy $G$. We plot the 3D surface $G(T, x_1)$ alongside the phase boundaries.


In [ ]:
G_np = -free_energy # The conjugate dual is exactly -G

T_np = to_numpy(T_grid)
x1_np = dual_coords[..., 1] # x1 is at index 1

fig3d = go.Figure()

# Plot the 3D Equilibrium Surface
fig3d.add_trace(go.Surface(
    x=x1_np,
    y=T_np,
    z=G_np,
    colorscale='Viridis',
    opacity=0.8,
    name='Gibbs Free Energy'
))

# Calculate the shadow floor level
min_G = G_np.min()
T_grid_floor = np.vstack([T_vals_np, T_vals_np])
z_floor = np.full_like(T_grid_floor, min_G)

# Solid Region Floor Shadow
fig3d.add_trace(go.Surface(
    x=np.vstack([np.zeros_like(x_left), x_left]),
    y=T_grid_floor,
    z=z_floor,
    colorscale=[[0, 'rgba(240,128,128,0.8)'], [1, 'rgba(240,128,128,0.8)']], # lightcoral
    showscale=False,
    name='Solid Region'
))

# Liquid Region Floor Shadow
fig3d.add_trace(go.Surface(
    x=np.vstack([x_right, np.ones_like(x_right)]),
    y=T_grid_floor,
    z=z_floor,
    colorscale=[[0, 'rgba(173,216,230,0.8)'], [1, 'rgba(173,216,230,0.8)']], # lightblue
    showscale=False,
    name='Liquid Region'
))

# Two-Phase Region Floor Shadow
fig3d.add_trace(go.Surface(
    x=np.vstack([x_left, x_right]),
    y=T_grid_floor,
    z=z_floor,
    colorscale=[[0, 'rgba(211,211,211,0.8)'], [1, 'rgba(211,211,211,0.8)']], # lightgray
    showscale=False,
    name='Two-Phase Region'
))

# Overlay the 3D boundary lines on the floor
shadow_z = np.full_like(T_vals_np, min_G)

fig3d.add_trace(go.Scatter3d(
    x=x_liquidus, y=T_vals_np, z=shadow_z,
    mode='lines', line=dict(color='blue', width=6),
    name='Liquidus (Floor)'
))

fig3d.add_trace(go.Scatter3d(
    x=x_solidus, y=T_vals_np, z=shadow_z,
    mode='lines', line=dict(color='red', width=6),
    name='Solidus (Floor)'
))

fig3d.update_layout(
    title='3D Gibbs Free Energy Surface & Phase Boundaries',
    scene=dict(
        xaxis_title='Mole Fraction (x1)',
        yaxis_title='Temperature (K)',
        zaxis_title='Gibbs Free Energy (J/mol)',
        xaxis=dict(range=[-0.05, 1.05]),
        yaxis=dict(range=[500, 2000]),
    ),
    width=900,
    height=800,
    margin=dict(l=0, r=0, b=0, t=50)
)

fig3d.show()
